# Guardian — training YOLOv5n `senjata_tajam` (tahap 1: data publik)

Panduan lengkap: `model/README.md`. Keputusan: K1 YOLOv5n 320, K13 satu kelas `senjata_tajam`.

**Sebelum mulai**
1. `Runtime → Change runtime type → T4 GPU`.
2. Simpan API key Roboflow di panel 🔑 *Secrets* (kiri) dengan nama `ROBOFLOW_API_KEY`, aktifkan *Notebook access*.
3. Isi daftar dataset di sel **2. Dataset**.

Semua hasil (dataset gabungan, run, model ONNX) disimpan di Google Drive `MyDrive/guardian/`, jadi aman
kalau Colab terputus. Jalankan sel dari atas ke bawah.

## 0. GPU + Google Drive

In [ ]:
!nvidia-smi -L
from google.colab import drive
drive.mount('/content/drive')
import os
ROOT = '/content/drive/MyDrive/guardian'
os.makedirs(ROOT, exist_ok=True)
print('ROOT =', ROOT)

## 1. YOLOv5

In [ ]:
%cd /content
!test -d yolov5 || git clone -q https://github.com/ultralytics/yolov5
%cd /content/yolov5
!pip install -q -r requirements.txt albumentations roboflow fiftyone onnx onnxruntime onnxsim

## 2. Dataset publik (Roboflow Universe)

Cari di https://universe.roboflow.com dengan kata kunci *knife*, *sickle*, *machete*, *scissors*, *sharp weapon*.
Pilih dataset yang **berupa deteksi objek (bounding box)**, gambarnya beragam (bukan satu benda di latar putih),
dan lisensinya jelas. Dari halaman dataset: *Download Dataset → YOLOv5 PyTorch → show download code*; salin
`workspace`, `project`, dan nomor `version`.

`keep`: nama kelas di dataset itu yang dianggap senjata tajam (huruf kecil). Kelas lain (mis. `gun`, `person`)
**dibuang kotaknya**. `None` = semua kelas dianggap senjata tajam.
Catat setiap dataset beserta lisensinya di `model/README.md`.

In [ ]:
DATASETS = [
    # dict(workspace='nama-workspace', project='nama-project', version=1, keep=['knife']),
]
N_NEG = 1500          # sampel COCO negatif sulit (sebelum disaring); 0 = tanpa negatif
VAL_FRAC_NEG = 0.15
assert DATASETS, 'isi DATASETS dulu'

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
rf = Roboflow(api_key=userdata.get('ROBOFLOW_API_KEY'))
RAW = '/content/raw'
os.makedirs(RAW, exist_ok=True)
for i, d in enumerate(DATASETS):
    loc = f"{RAW}/d{i}_{d['project']}"
    if not os.path.isdir(loc):
        rf.workspace(d['workspace']).project(d['project']).version(d['version']).download('yolov5', location=loc)
    d['dir'] = loc
print([d['dir'] for d in DATASETS])

### 2b. Cek nama kelas tiap dataset
Cocokkan `keep` di sel 2 dengan nama yang tercetak di sini. Kalau `keep` salah, perbaiki sel 2, jalankan ulang
sel 2 (cepat, dataset tidak diunduh ulang), lalu sel ini lagi. Jangan lanjut ke sel 3 sebelum semua baris `OK`.

In [ ]:
import yaml
for i, d in enumerate(DATASETS):
    y = yaml.safe_load(open(f"{d['dir']}/data.yaml"))
    n = y['names']
    names = [str(n[k]) for k in sorted(n)] if isinstance(n, dict) else [str(x) for x in n]
    keep = d.get('keep')
    hit = names if keep is None else [x for x in names if x.lower() in {k.lower() for k in keep}]
    status = 'OK' if hit else '!! keep tidak cocok dengan kelas mana pun'
    print(f"d{i} {d['project']}: kelas={names}  keep={keep}  -> dipakai={hit}  {status}")

## 3. Gabungkan jadi satu kelas `senjata_tajam`

- Indeks kelas dipetakan ulang ke `0`; label poligon (segmentasi) diubah ke kotak.
- `train` → train, `valid` + `test` → val (tahap 1). Test set sebenarnya nanti **hanya data primer kacamata** (plan 9.1).

In [ ]:
import shutil, random, yaml, glob
from pathlib import Path

OUT = Path(f'{ROOT}/dataset_s1')
if OUT.exists():
    shutil.rmtree(OUT)
for s in ('train', 'val'):
    (OUT / s / 'images').mkdir(parents=True)
    (OUT / s / 'labels').mkdir(parents=True)

def names_of(ds_dir):
    y = yaml.safe_load(open(f'{ds_dir}/data.yaml'))
    n = y['names']
    return [str(n[k]) for k in sorted(n)] if isinstance(n, dict) else [str(x) for x in n]

def convert_line(parts):
    vals = list(map(float, parts[1:]))
    if len(vals) == 4:
        return vals
    xs, ys = vals[0::2], vals[1::2]          # poligon -> kotak
    x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
    return [(x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1]

stats = {}
for i, d in enumerate(DATASETS):
    names = names_of(d['dir'])
    keep = set(range(len(names))) if d.get('keep') is None else {
        k for k, n in enumerate(names) if n.lower() in {x.lower() for x in d['keep']}}
    print(f"d{i} {d['project']}: kelas {names} -> dipakai {[names[k] for k in sorted(keep)]}")
    for src, dst in (('train', 'train'), ('valid', 'val'), ('test', 'val')):
        for img in glob.glob(f"{d['dir']}/{src}/images/*"):
            stem = Path(img).stem
            lbl = Path(f"{d['dir']}/{src}/labels/{stem}.txt")
            rows = []
            if lbl.exists():
                for line in lbl.read_text().splitlines():
                    p = line.split()
                    if len(p) >= 5 and int(float(p[0])) in keep:
                        rows.append('0 ' + ' '.join(f'{v:.6f}' for v in convert_line(p)))
            name = f'd{i}_{stem}'
            shutil.copy(img, OUT / dst / 'images' / (name + Path(img).suffix))
            (OUT / dst / 'labels' / f'{name}.txt').write_text('\n'.join(rows))
            key = (dst, 'pos' if rows else 'neg')
            stats[key] = stats.get(key, 0) + 1
print(stats)

### 3b. Negatif sulit dari COCO

Gambar COCO yang berisi benda mirip (ponsel, sendok, garpu, remote, sikat gigi, payung, botol) **tanpa** pisau
atau gunting, dimasukkan dengan label kosong untuk menekan false positive (plan 9, 9A.3).

In [ ]:
if N_NEG:
    import fiftyone.zoo as foz
    HARD = ['cell phone', 'spoon', 'fork', 'remote', 'toothbrush', 'umbrella', 'bottle']
    coco = foz.load_zoo_dataset('coco-2017', split='validation', label_types=['detections'],
                                classes=HARD, max_samples=N_NEG, shuffle=True, seed=0)
    random.seed(0)
    n = {'train': 0, 'val': 0}
    for s in coco:
        labels = {det.label for det in (s.ground_truth.detections if s.ground_truth else [])}
        if labels & {'knife', 'scissors'}:
            continue
        dst = 'val' if random.random() < VAL_FRAC_NEG else 'train'
        name = 'neg_' + Path(s.filepath).stem
        shutil.copy(s.filepath, OUT / dst / 'images' / (name + '.jpg'))
        (OUT / dst / 'labels' / f'{name}.txt').write_text('')
        n[dst] += 1
    print('negatif COCO:', n)

In [ ]:
DATA_YAML = f'{OUT}/data.yaml'
yaml.safe_dump({'path': str(OUT), 'train': 'train/images', 'val': 'val/images',
                'nc': 1, 'names': ['senjata_tajam']}, open(DATA_YAML, 'w'), sort_keys=False)
for s in ('train', 'val'):
    labels = list((OUT / s / 'labels').glob('*.txt'))
    pos = sum(1 for f in labels if f.read_text().strip())
    print(f'{s}: {len(labels)} gambar, {pos} positif, {len(labels) - pos} negatif')
print(open(DATA_YAML).read())

### 3c. Cek visual label (wajib sekilas)
Kotak harus rapat di benda tajam. Kalau banyak yang salah, buang dataset itu dari `DATASETS`.

In [ ]:
import cv2, matplotlib.pyplot as plt
imgs = random.sample([p for p in (OUT / 'train' / 'images').iterdir()], 12)
fig, axs = plt.subplots(3, 4, figsize=(16, 12))
for ax, p in zip(axs.flat, imgs):
    im = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = im.shape[:2]
    for line in (OUT / 'train' / 'labels' / (p.stem + '.txt')).read_text().splitlines():
        _, cx, cy, bw, bh = map(float, line.split())
        cv2.rectangle(im, (int((cx - bw / 2) * w), int((cy - bh / 2) * h)),
                      (int((cx + bw / 2) * w), int((cy + bh / 2) * h)), (255, 0, 0), max(2, w // 200))
    ax.imshow(im); ax.set_title(p.name[:30], fontsize=8); ax.axis('off')
plt.tight_layout()

## 4. Hyperparameter augmentasi

Dasar `hyp.scratch-low.yaml`, dengan variasi warna/kecerahan lebih kuat dan skala lebih lebar agar mendekati
kamera kacamata (OV3660, JPEG QVGA, cahaya berubah-ubah). `albumentations` yang terpasang otomatis menambah
blur, kompresi JPEG, dan CLAHE ringan.

In [ ]:
hyp = yaml.safe_load(open('data/hyps/hyp.scratch-low.yaml'))
hyp.update(hsv_h=0.02, hsv_s=0.8, hsv_v=0.5, scale=0.6, degrees=5.0, fliplr=0.5, mixup=0.05)
HYP = f'{ROOT}/hyp_guardian.yaml'
yaml.safe_dump(hyp, open(HYP, 'w'))
print(hyp)

## 5. Training

±1–3 jam di T4 untuk beberapa ribu gambar. Kalau Colab terputus, jalankan sel **5b** (resume).

In [ ]:
RUN_NAME = 'weapon_v5n_s1'
!python train.py --img 320 --batch 64 --epochs 150 --patience 30 \
  --data {DATA_YAML} --hyp {HYP} --weights yolov5n.pt --cache ram \
  --project {ROOT}/runs --name {RUN_NAME} --exist-ok

In [ ]:
# 5b. resume setelah terputus (jalankan sel 0, 1, lalu ini saja)
# RUN_NAME = 'weapon_v5n_s1'
# !python train.py --resume {ROOT}/runs/{RUN_NAME}/weights/last.pt

## 6. Evaluasi

Catat **P, R, mAP50, mAP50-95** dari baris `all` ke `docs/PROGRESS.md`. Lihat juga `PR_curve.png` dan
`confusion_matrix.png`. Untuk sistem ini **recall lebih penting**; false positive diredam filter temporal 3/5 frame.

In [ ]:
W = f'{ROOT}/runs/{RUN_NAME}/weights/best.pt'
!python val.py --weights {W} --data {DATA_YAML} --img 320 --conf 0.001 --iou 0.6 --task val
from IPython.display import Image, display
for f in ('results.png', 'PR_curve.png', 'confusion_matrix.png'):
    display(Image(f'{ROOT}/runs/{RUN_NAME}/{f}', width=900))

## 7. Ekspor ONNX 320 (format yang dibaca `sbc/guardian/vision/detector.py`)

In [ ]:
!python export.py --weights {W} --include onnx --img 320 --opset 12 --simplify
import onnxruntime as ort, numpy as np
ONNX = W.replace('.pt', '.onnx')
s = ort.InferenceSession(ONNX)
i, o = s.get_inputs()[0], s.get_outputs()[0]
print(i.name, i.shape, '->', o.name, o.shape)        # harapan: [1, 3, 320, 320] -> [1, 6300, 6]
assert list(i.shape) == [1, 3, 320, 320] and o.shape[-1] == 6
out = s.run(None, {i.name: np.random.rand(1, 3, 320, 320).astype(np.float32)})[0]
print('ok', out.shape)
FINAL = f'{ROOT}/yolov5n_weapon.onnx'
shutil.copy(ONNX, FINAL)
print('salin ke Orange Pi:', FINAL)

## 8. Selanjutnya (di Orange Pi) — lihat `model/README.md` langkah 5
Unduh `MyDrive/guardian/yolov5n_weapon.onnx`, salin ke `sbc/models/`, lalu jalankan `tools/bench_detector.py`.